# Chapitre 12 - Exercices 12.6 à 12.10

Ce notebook traite les exercices 12.6 à 12.10, centrés sur la justesse de bout en
bout :

- **12.6** : identifiants de requête : relances, collisions et faux doublons (calcul et
  mini-lab) ;
- **12.7** : deux contraintes d'unicité à la fois, sans transaction (mini-lab) ;
- **12.8** : virement multi-partition : injecter une panne à chaque étape (mini-lab et
  calcul) ;
- **12.9** : combien de billets vendre en trop ? (calcul) ;
- **12.10** : comparer deux répliques avec un arbre de Merkle (calcul et mini-lab).

In [1]:
import math
import random
from math import comb

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import dataflow as dfl
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET, COLORS,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 4)
SHARD, SHARDS = "partition", "partitions"

## Exercice 12.6 : identifiants de requête : relances, collisions et faux doublons

**Énoncé.**

1. Un client renvoie sa requête tant qu'il ne reçoit pas de réponse, jusqu'à $m$ envois ;
   chaque réponse se perd indépendamment avec probabilité $p$ (la requête, elle, est
   toujours exécutée). Calculer le nombre moyen d'exécutions par requête sans
   déduplication. Application : $p = 0{,}1$, $m = 5$.
2. On identifie les requêtes par 64 bits aléatoires, à raison d'un milliard de requêtes
   par jour, conservés un an pour la déduplication. Quelle est la probabilité d'au moins
   une collision ? Et avec les 122 bits aléatoires d'un UUID v4 ?
3. Variante : l'identifiant est le hachage des champs du formulaire (compte source,
   bénéficiaire, montant, date). Chaque utilisateur fait 3 virements par jour vers l'un de
   ses 5 bénéficiaires habituels (au hasard), d'un montant pris parmi 10, 20, 50 et
   100 EUR (avec probabilités 0,4 ; 0,3 ; 0,2 ; 0,1). Quelle fraction des virements
   légitimes est supprimée à tort ? Simuler sur 100 000 utilisateurs.

**Rappels.** Une relance traverse TCP, la base et le 2PC ; seul un identifiant de bout en
bout la reconnaît (cours 4, sections 2 et 3 ; sections *Duplicate suppression* et
*Operation identifiers*).

**Solution.**

*1.* Le $i$-ième envoi a lieu si les $i - 1$ réponses précédentes se sont perdues :
$E = \sum_{i=1}^{m} p^{i-1} = \frac{1 - p^m}{1 - p}$. Pour $p = 0{,}1$, $m = 5$ :
$E = 0{,}99999/0{,}9 \approx 1{,}111$ : **11,1 %** d'exécutions en trop.

*2.* $n = 3{,}65 \cdot 10^{11}$ ; $n^2 / 2^{65} \approx 1{,}33 \cdot 10^{23} / 3{,}69 \cdot
10^{19} \approx 3\,600$, donc $P \approx 1 - e^{-3600} = 1$ : une collision est
**certaine** (sur une seule journée, déjà 2,7 %). Avec 122 bits :
$1{,}33 \cdot 10^{23} / 2^{123} \approx 1{,}3 \cdot 10^{-14}$, négligeable.

*3.* Pour un utilisateur, les 3 virements tombent dans des cases (bénéficiaire, montant) ;
un virement est supprimé à tort s'il tombe dans une case déjà occupée ce jour-là. Le
nombre attendu de cases distinctes est $\sum_c [1 - (1 - q_c)^3]$, où $q_c$ est la
probabilité de la case ; les virements supprimés sont $3 - E[\text{distinctes}]$. Avec
des montants uniformes (20 cases équiprobables), on trouverait $3 - 20(1 - 0{,}95^3)
\approx 0{,}15$ par utilisateur, soit 4,9 % ; avec des montants concentrés sur les petites
valeurs, c'est davantage (calcul ci-dessous).

**Vérification.**

In [2]:
p, m = 0.1, 5
E = (1 - p ** m) / (1 - p)
sim = dfl.retrying_client(dfl.make_bank_db({1: 0.0, 2: 1e7}), 20_000, p, use_ids=False, seed=2)
print(f"exécutions par requête : formule {E:.4f} ; simulation {sim['credited'] / sim['intended']:.4f}")
print(f"collision 64 bits sur un an : {dfl.uuid_collision_prob(3.65e11, 64):.6f} ; sur un jour : {dfl.uuid_collision_prob(1e9, 64):.4f} ; "
      f"122 bits sur un an : {dfl.uuid_collision_prob(3.65e11, 122):.1e}")
q_amount = np.array([0.4, 0.3, 0.2, 0.1])
q = np.outer(np.full(5, 0.2), q_amount).ravel()           # 20 cases (bénéficiaire, montant)
attendu = 3 - np.sum(1 - (1 - q) ** 3)
rng = np.random.default_rng(5)
n_users = 100_000
cases = rng.choice(20, size=(n_users, 3), p=q)
distinct = np.array([len(set(row)) for row in cases])
print(f"virements légitimes supprimés à tort : calcul {attendu / 3:.2%} ; simulation {(3 - distinct).sum() / (3 * n_users):.2%}")

exécutions par requête : formule 1.1111 ; simulation 1.1098
collision 64 bits sur un an : 1.000000 ; sur un jour : 0.0267 ; 122 bits sur un an : 1.3e-14
virements légitimes supprimés à tort : calcul 5.87% ; simulation 5.90%


Le hachage des champs supprime à tort **près de 6 %** des virements légitimes dans ce
scénario : beaucoup trop. Il faut un élément propre à chaque intention, par exemple un
jeton aléatoire généré à l'affichage du formulaire (et donc identique pour toutes les
relances de **ce** formulaire, mais différent pour le formulaire suivant).

> **À retenir.** Sans déduplication, un taux de réponses perdues $p$ coûte environ
> $p/(1-p)$ d'exécutions en trop. Un identifiant aléatoire doit être long (64 bits ne
> suffisent pas à l'échelle d'un an) ; un identifiant dérivé du contenu confond des
> intentions distinctes.